In [1]:
import numpy as np


W = [w1 w2 w3]

b = scalar number

X = [x1 x2 x3]

### multiplication without **vectorization**

f(w,b) = w1x1 + w2x2 + w3x3 + b

i.e:


f(w,b) = w[0] * x[0] +

         w[1] * x[1] +

         w[2] * x[2] + b


In [2]:
"""
for j in range(0,n):
    f_wb = w[j]*x[j] + b
    return f_wb
"""

'\nfor j in range(0,n):\n    f_wb = w[j]*x[j] + b\n    return f_wb\n'

#### **With Vectorization**

f_wb = np.dot(w,x) + b

---

### **1) Gradient Calculation without linear algebra:**



In [3]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

In [4]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])

In [5]:
class Calculations():
    def __init__(self,x,y,b,w):
        self.x = x
        self.y = y
        self.b = b
        self.w = w
    def gradient(self):
        m = self.x.shape[0]
        n = self.x.shape[1]
        dj_dw = np.zeros(len(self.w))
        dj_db = 0
        for i in range(m):
            error = (np.dot(self.x[i],self.w)+self.b) - self.y[i]
            for j in range(n):
                dj_dw = dj_dw + np.sum(error) *self.x[i,j]
            dj_db = dj_db + np.sum(error)
        dj_dw = 1/m*dj_dw
        dj_db = 1/m*dj_db
        return dj_dw,dj_db
        


In [6]:
z = Calculations(X_train,y_train,b_init,w_init)
z.gradient()

(array([-0.00280397, -0.00280397, -0.00280397, -0.00280397]),
 np.float64(-1.673925169143331e-06))

### **2) Gradient Calculation WITH linear algebra:**



starting from the partial derivation:

- dj_dw = sum(error)* x[i,j]

- dj_db = sum(error)


w = w - alpha*(d/dw*J(w,b))

b = b - alpha * (d/db*J(w,b))

Calculating dW, starting from the **scalar** form:

dW = 1/m * sum(y_hat - y) * x[i,j]



1) dW in matrix representation written as [dJ/dw1,dJ/dw2,dJ/w3,dJ/w4..] for X [x1 x2 x3 x4]

2) error(i) = y_hat(i)- y(i)

error [error(1) error(2) error(3)] in matrix form

### If i put all these into formula dWj = 1/m  sum ( error(i) ) * x [ i , j ]

#### Shape of error = m*1 

#### Shape of **X** = m * n


X^T = n * m = > Inner dimension match and we can multiply now so:

dWj = 1/m * X^T * error 

#### __Finall form:__

### __dWj = 1/m * (X^T @ (XW + b - y))__

In [7]:
class vectorized_form():
    def __init__(self,x,y,b,w):
        self.x = x
        self.y = y
        self.b = b
        self.w = w
    def get_gradients(self):
        m = self.x.shape[0]
        self.m = m
        error = np.dot(self.x,self.w)+self.b - self.y
        dj_dw = 1/m*(self.x.T@error)
        dj_db = 1/m* np.sum(error)
        return dj_dw,dj_db
    def cost_function(self,w=None,b=None):
        if w is None:
            w = self.w
        if b is None:
            b = self.b

        error = self.x@w + b - self.y
        error_sq = np.sum(error**2)
        j_cost = 1/(2*self.m)*(error_sq)
        return j_cost

    def descent(self,num_iter,alpha):
        history = []
        cost_before = self.cost_function()
        #print(cost_before)
        for i in range(num_iter):
            dj_dw,dj_db = self.get_gradients()
            self.w = self.w - alpha * dj_dw
            self.b = self.b - alpha * dj_db
            J_cost = self.cost_function(self.w,self.b)
            if abs(J_cost - cost_before) < 0.01:
                print(f"Conversion achieved in iteration num {i}. Difference {cost_before-J_cost}")
                break
            history.append({'cost_history':J_cost,'w_history':self.w.copy(),'b_history':self.b.copy()})
        return history

---
### Before I proceed to calculate gradinet descnet i must scale the parameters 

#### Here I implement the standard scaler from scratch 

#### **Standard Scaler** 

X = x - mean(x) / std

mean = 0 
std = 1

mean = sum(x)/len(x)

std = math.sqrt(1/len(x)*(x - mu)**2)

In [8]:
import math
class Scaler():
    def __init__(self):
        pass
    def fit(self,x):
        self.x=x
        m,n = self.x.shape

        meanz = np.zeros(n,)
        stdz = np.zeros(n,)
        
        for i in range(m):
            for j in range(n):
                meanz[j] += self.x[i][j]
        meanz = meanz/m
        self.meanz=meanz
        
        for i in range(m):
            for j in range(n):
                stdz[j] += (self.x[i,j]-meanz[j])**2
        
        stdz = stdz/m + np.exp(0.05)
        stdz = np.sqrt(stdz)
        self.stdz=stdz



    def transform(self,x_new):
        res = (x_new - self.meanz) / self.stdz
        return res


#### Little testing:

In [9]:
zz =np.random.randint(low=0.00995,high=10000,size=(15,4))
zz

array([[7390, 6137, 8891, 2092],
       [4568,  923, 7435, 7641],
       [9820, 7486, 4547, 2377],
       [4494, 3396, 5875, 9292],
       [7165, 1687, 4718, 4971],
       [9578,  746, 1209, 4545],
       [ 516, 8877,  203,  212],
       [5188, 7392, 4944, 4468],
       [7236, 4172, 1421, 8435],
       [ 196, 4621, 6029, 1677],
       [3979, 6389, 3117, 4826],
       [6811, 3043, 3762, 7455],
       [8000, 8005,  852, 1750],
       [8795, 4049, 8978, 3071],
       [ 847, 5709,  625, 3620]], dtype=int32)

In [10]:
scale = Scaler()
scale.fit(zz)

scaled_x = scale.transform(zz)
print(scaled_x)
print(scaled_x.shape)

[[ 0.56893868  0.51962694  1.66515608 -0.88601069]
 [-0.3479218  -1.57273897  1.15120013  1.21792347]
 [ 1.35843938  1.06097746  0.13176004 -0.77795136]
 [-0.37196421 -0.5803299   0.60053305  1.84390928]
 [ 0.49583676 -1.26614757  0.19212163  0.20557814]
 [ 1.27981421 -1.64376865 -1.04652632  0.04405788]
 [-1.6644061   1.61918248 -1.40163599 -1.59882313]
 [-0.14648541  1.02325548  0.27189776  0.0148629 ]
 [ 0.51890448 -0.26892293 -0.97169208  1.51897297]
 [-1.76837327 -0.08874029  0.65489377 -1.04336025]
 [-0.53928638  0.62075395 -0.37301812  0.15060059]
 [ 0.38082308 -0.72198796 -0.14533846  1.14740053]
 [ 0.7671261   1.26925094 -1.17254436 -1.01568189]
 [ 1.02541954 -0.31828254  1.69586636 -0.51481741]
 [-1.55686506  0.34787155 -1.25267349 -0.30666101]]
(15, 4)


---

##### Contine with gradient descent

In [11]:
scaler = Scaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
print(X_train_scaled)
print(X_train.shape,X_train_scaled.shape)

[[ 1.26311253  1.0322685  -0.29537954  1.18785497]
 [-0.08073503 -0.2064537   0.59075908  0.        ]
 [-1.1823775  -0.8258148  -0.29537954 -1.18785497]]
(3, 4) (3, 4)


In [12]:
v = vectorized_form(X_train_scaled,y_train,b_init,w_init)
gradients = v.get_gradients()
v.cost_function()
desc_history = v.descent(1000,0.005)
cost_only = []
for i in desc_history:
    cost_only.append(i.get('cost_history'))

In [13]:
best_w = 0
for i in desc_history:
    best_w = i.get('w_history')[-1]

In [16]:
def get_best_params(paramlist):
    b = 0
    w = 0
    for i in paramlist: 
        w = i.get('w_history')[-1]
        b = i.get('b_history')
    return w,b

In [17]:
best_w,best_b = get_best_params(desc_history)

In [18]:
print(best_w,best_b)

20.666973036891722 293.29491972509356
